# 09 — Unsupervised Learning

**Topics:** KMeans, DBSCAN, PCA, dimensionality reduction, cluster evaluation.

**Reference:** [sklearn clustering](https://scikit-learn.org/stable/modules/clustering.html) | [sklearn decomposition](https://scikit-learn.org/stable/modules/decomposition.html)

**Dataset:** Wholesale customers — B2B spending data across product categories.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Wholesale customers dataset: annual spending in 6 product categories
wholesale_raw = fetch_openml(name='wholesale-customers', version=1, as_frame=True, parser='auto').frame
wholesale = wholesale_raw.copy()
wholesale.columns = [c.strip().lower().replace(' ', '_') for c in wholesale.columns]

# Feature matrix — drop categorical region/channel
feature_cols = ['fresh', 'milk', 'grocery', 'frozen', 'detergents_paper', 'delicassen']
X = wholesale[feature_cols].copy()
print(wholesale.shape)
X.describe()

---
## Exercise 1 — PCA: Variance Explained

**Task:** Use PCA to understand the intrinsic dimensionality of this dataset.

1. Standardize `X` with `StandardScaler`.
2. Fit PCA (all 6 components) on the scaled data.
3. Build `pca_summary`: DataFrame with columns `component`, `explained_variance_ratio`, `cumulative_variance`.
4. Find `n_components_95`: the minimum number of components needed to explain 95% of variance.
5. Build `loadings_df`: the PCA loading matrix — rows = original features, columns = PC1..PC6. Rounded to 4dp.

In [ ]:
def analyze_pca(X: pd.DataFrame):
    """
    Returns (pca_fitted, X_scaled, pca_summary, n_components_95, loadings_df)
    """
    # YOUR CODE HERE
    pass

pca, X_scaled, pca_summary, n_components_95, loadings_df = analyze_pca(X)

In [ ]:
# --- ASSERTIONS ---
assert list(pca_summary.columns) == ['component', 'explained_variance_ratio', 'cumulative_variance']
assert len(pca_summary) == 6
assert abs(pca_summary['cumulative_variance'].iloc[-1] - 1.0) < 1e-6, "Cumulative variance must sum to 1"
assert pca_summary['cumulative_variance'].is_monotonic_increasing
assert isinstance(n_components_95, (int, np.integer))
assert n_components_95 <= 6

assert loadings_df.shape == (6, 6)
assert list(loadings_df.index) == feature_cols

print(f"✓ Exercise 1 passed — {n_components_95} components explain 95% of variance")
print(pca_summary)

---
## Exercise 2 — PCA Projection & Reconstruction Error

**Task:** Project data into reduced space and measure information loss.

1. Project `X_scaled` to `n_components_95` dimensions → `X_reduced`. Shape: `(n_samples, n_components_95)`.
2. Reconstruct back to original space → `X_reconstructed`. Shape: `(n_samples, 6)`.
3. Compute `reconstruction_error_per_sample`: MSE between `X_scaled` and `X_reconstructed` for each row. Shape: `(n_samples,)`.
4. Compute `mean_reconstruction_error` (scalar).
5. Find `outlier_indices`: indices of the top 5% of samples by reconstruction error — potential anomalies.

In [ ]:
def pca_project_reconstruct(pca_model, X_scaled, n_components):
    """
    Returns (X_reduced, X_reconstructed, reconstruction_error_per_sample,
             mean_reconstruction_error, outlier_indices)
    """
    # YOUR CODE HERE
    pass

(X_reduced, X_reconstructed, reconstruction_error_per_sample,
 mean_reconstruction_error, outlier_indices) = pca_project_reconstruct(pca, X_scaled, n_components_95)

In [ ]:
# --- ASSERTIONS ---
assert X_reduced.shape == (len(X), n_components_95)
assert X_reconstructed.shape == X_scaled.shape
assert reconstruction_error_per_sample.shape == (len(X),)
assert (reconstruction_error_per_sample >= 0).all()
assert mean_reconstruction_error > 0
expected_outlier_count = int(np.ceil(len(X) * 0.05))
assert len(outlier_indices) == expected_outlier_count, f"Expected top 5% = {expected_outlier_count} outliers"
print(f"✓ Exercise 2 passed — Mean reconstruction error: {mean_reconstruction_error:.6f}")
print(f"Top outlier indices: {outlier_indices[:5]}")

---
## Exercise 3 — KMeans: Elbow Method & Optimal K

**Task:** Find the optimal number of clusters using the elbow method and silhouette scores.

1. Run KMeans for k = 2 through 10 on `X_scaled`. Use `random_state=42`, `n_init=10`.
2. For each k, compute: `inertia`, `silhouette_score`, `davies_bouldin_score`.
3. Return `kmeans_eval`: DataFrame with columns `k`, `inertia`, `silhouette`, `davies_bouldin`.
4. Find `best_k_silhouette`: k with highest silhouette score.
5. Find `best_k_db`: k with lowest Davies-Bouldin score.
6. Do the two metrics agree? Note your observation in markdown.

In [ ]:
def evaluate_kmeans(X_scaled, k_range):
    """
    Returns (kmeans_eval, best_k_silhouette, best_k_db)
    """
    # YOUR CODE HERE
    pass

kmeans_eval, best_k_silhouette, best_k_db = evaluate_kmeans(X_scaled, range(2, 11))

In [ ]:
# --- ASSERTIONS ---
assert list(kmeans_eval.columns) == ['k', 'inertia', 'silhouette', 'davies_bouldin']
assert len(kmeans_eval) == 9
assert kmeans_eval['inertia'].is_monotonic_decreasing, "Inertia must decrease as k increases"
assert kmeans_eval['silhouette'].between(0, 1).all()
assert 2 <= best_k_silhouette <= 10
assert 2 <= best_k_db <= 10
print(f"✓ Exercise 3 passed — Best k (silhouette): {best_k_silhouette} | Best k (DB): {best_k_db}")
print(kmeans_eval)

---
## Exercise 4 — KMeans Cluster Profiling

**Task:** After choosing k, interpret what the clusters mean in business terms.

Using `best_k_silhouette`:

1. Fit final KMeans on `X_scaled`, add `cluster` labels to the original `X` (unscaled).
2. Build `cluster_profile`: mean of each original feature per cluster. Shape: `(k, 6)`. Index = cluster label.
3. Add column `cluster_size`: count of samples in each cluster.
4. Add column `cluster_pct`: percentage of total samples. Rounded to 2dp.
5. Rank clusters by `fresh` spending descending — add `rank_by_fresh` column.
6. Write a one-sentence business interpretation per cluster in a markdown cell.

In [ ]:
def profile_clusters(X_original, X_scaled, k):
    """
    Returns (X_with_labels, cluster_profile)
    """
    # YOUR CODE HERE
    pass

X_labeled, cluster_profile = profile_clusters(X, X_scaled, best_k_silhouette)

In [ ]:
# --- ASSERTIONS ---
assert 'cluster' in X_labeled.columns
assert X_labeled['cluster'].nunique() == best_k_silhouette
for col in ['cluster_size', 'cluster_pct', 'rank_by_fresh']:
    assert col in cluster_profile.columns, f"Missing: {col}"
assert abs(cluster_profile['cluster_pct'].sum() - 100.0) < 0.1
assert cluster_profile['cluster_size'].sum() == len(X)
print("✓ Exercise 4 passed")
print(cluster_profile)

**Business interpretation:**
- Cluster 0: *...*
- Cluster 1: *...*

---
## Exercise 5 — DBSCAN: Density-Based Clustering

**Task:** Use DBSCAN to find clusters of arbitrary shape and flag anomalies (noise points).

DBSCAN labels noise points as `-1`.

1. Run DBSCAN on `X_scaled` with `eps=1.5`, `min_samples=5`.
2. Report: `n_clusters` (excluding noise), `n_noise_points`, `noise_pct`.
3. Tune DBSCAN: try `eps` values `[0.5, 1.0, 1.5, 2.0, 2.5]` with `min_samples=5`. For each, record `n_clusters`, `n_noise_points`, and silhouette score (skip if all noise or only 1 cluster).
4. Return `dbscan_tuning`: DataFrame with columns `eps`, `n_clusters`, `n_noise`, `silhouette`.
5. Return `best_eps`: eps producing highest silhouette (among runs with ≥ 2 clusters).

In [ ]:
def tune_dbscan(X_scaled, eps_values, min_samples=5):
    """
    Returns (dbscan_tuning DataFrame, best_eps)
    """
    # YOUR CODE HERE
    pass

dbscan_tuning, best_eps = tune_dbscan(X_scaled, [0.5, 1.0, 1.5, 2.0, 2.5])

In [ ]:
# --- ASSERTIONS ---
assert list(dbscan_tuning.columns) == ['eps', 'n_clusters', 'n_noise', 'silhouette']
assert len(dbscan_tuning) == 5
assert best_eps in [0.5, 1.0, 1.5, 2.0, 2.5]
print(f"✓ Exercise 5 passed — Best eps: {best_eps}")
print(dbscan_tuning)

---
## Exercise 6 — PCA + Clustering Pipeline

**Task:** Combine dimensionality reduction with clustering — a standard production pattern.

1. Build a pipeline: `StandardScaler` → `PCA(n_components=2)` → `KMeans(n_clusters=best_k_silhouette, random_state=42)`.
2. Fit on `X`.
3. Compare silhouette scores: clustering in full scaled space vs PCA-reduced space. Which is better?
4. Build `pca2d_df`: DataFrame with columns `PC1`, `PC2`, `cluster` for visualization readiness.
5. Compute `cluster_separation`: for each pair of clusters, the Euclidean distance between their centroids in PCA space. Return as a `(k × k)` DataFrame.

In [ ]:
def pca_clustering_pipeline(X, k):
    """
    Returns (pipeline, pca2d_df, cluster_separation, silhouette_full, silhouette_pca)
    """
    # YOUR CODE HERE
    pass

pca_pipe, pca2d_df, cluster_separation, sil_full, sil_pca = pca_clustering_pipeline(X, best_k_silhouette)

In [ ]:
# --- ASSERTIONS ---
assert list(pca2d_df.columns) == ['PC1', 'PC2', 'cluster']
assert len(pca2d_df) == len(X)
assert cluster_separation.shape == (best_k_silhouette, best_k_silhouette)
assert (np.diag(cluster_separation.values) == 0).all(), "Distance from a cluster to itself is 0"
assert 0 < sil_full < 1 and 0 < sil_pca < 1
print(f"✓ Exercise 6 passed")
print(f"Silhouette — Full space: {sil_full:.4f} | PCA-2D: {sil_pca:.4f}")
print(cluster_separation)

---
## Exercise 7 — Silhouette Analysis Per Sample

**Task:** Go beyond the mean silhouette — diagnose cluster quality at the sample level.

Using the best KMeans result (full scaled space):

1. Compute `silhouette_vals`: silhouette score for each individual sample using `silhouette_samples`.
2. Build `silhouette_df`: columns `cluster`, `silhouette_value`, sorted by cluster then silhouette descending.
3. Compute per-cluster stats: `mean_silhouette`, `pct_below_avg` (% of samples in that cluster below the overall mean silhouette).
4. Return `cluster_silhouette_stats`: DataFrame indexed by cluster label.
5. Identify `weak_clusters`: list of clusters where `mean_silhouette < 0.3` — candidates for merging or investigation.

In [ ]:
def analyze_silhouettes(X_scaled, labels):
    """
    Returns (silhouette_df, cluster_silhouette_stats, weak_clusters)
    """
    # YOUR CODE HERE
    pass

# Get labels from best KMeans
km_best = KMeans(n_clusters=best_k_silhouette, random_state=42, n_init=10).fit(X_scaled)
silhouette_df, cluster_silhouette_stats, weak_clusters = analyze_silhouettes(X_scaled, km_best.labels_)

In [ ]:
# --- ASSERTIONS ---
assert list(silhouette_df.columns) == ['cluster', 'silhouette_value']
assert len(silhouette_df) == len(X)
assert list(cluster_silhouette_stats.columns) == ['mean_silhouette', 'pct_below_avg']
assert cluster_silhouette_stats.index.name == 'cluster' or 'cluster' in str(cluster_silhouette_stats.index.name)
assert isinstance(weak_clusters, list)
print(f"✓ Exercise 7 passed")
print(cluster_silhouette_stats)
print(f"Weak clusters: {weak_clusters}")